# Agentic RAG & Multi-Agent Intelligent Workflows
## HR Policy Email Triage Assistant using Agnos/Agno + Zapier

**Use case:** An HR Policy Email Triage Assistant that retrieves company policy information, reasons over employee requests, and triggers workflow actions such as drafting replies, logging requests, creating approval tasks, or escalating urgent security issues.

**Core idea:** Standard LLMs generate text. This system retrieves evidence first, uses specialized agents to reason over the evidence, and then routes the request to a business workflow.


## HR Policy Email Triage Assistant using Agnos/Agno + Zapier

**Use case:** An HR Policy Email Triage Assistant that retrieves company policy information, reasons over employee requests, and triggers workflow actions such as drafting replies, logging requests, creating approval tasks, or escalating urgent security issues.

**Core idea:** Standard LLMs generate text. This system retrieves evidence first, uses specialized agents to reason over the evidence, and then routes the request to a business workflow.

## Task 1 - Use Case and Importance

Organizations lose time when employees repeatedly email HR about sick leave, remote work, overtime, vacation requests, or security concerns. Manual triage creates delays, inconsistent answers, and missed escalations. This project builds an Agentic RAG workflow where a multi-agent system retrieves relevant HR policy evidence, reasons over the request, and then chooses an action. Agnos/Agno is used for multi-agent coordination, while Zapier is used for no-code workflow execution. The value is improved accuracy, faster response time, better compliance, and reduced workload for HR staff.

#Task 2 — Building a Multi-Agent System Using Agnos/Agno

For Task 2, we will build three specialized agents plus one coordinator. This satisfies the requirement for 2+ agents, retrieval from multiple sources, cross-agent communication, and a final synthesized output.

Agno supports tool-using agents, including DuckDuckGo search tools and custom Python tools, which fits our HR Policy Assistant design

In [7]:
!pip install -q agno openai duckduckgo-search ddgs scikit-learn pandas numpy

In [8]:
import os
import getpass

# Clear the wrong/old key from the notebook session
os.environ.pop("OPENAI_API_KEY", None)

# Paste your real OpenAI API key from the OpenAI API platform
api_key = getpass.getpass("Paste your OpenAI API key here: ").strip()

os.environ["OPENAI_API_KEY"] = api_key

print("API key saved for this Colab session.")
print("Masked key preview:", os.environ["OPENAI_API_KEY"][:7] + "..." + os.environ["OPENAI_API_KEY"][-4:])

Paste your OpenAI API key here: ··········
API key saved for this Colab session.
Masked key preview: sk-proj...8CoA


In [9]:
import os

print("OPENAI_API_KEY exists:", "OPENAI_API_KEY" in os.environ)
print("Length:", len(os.environ["OPENAI_API_KEY"]))
print("Preview:", os.environ["OPENAI_API_KEY"][:7] + "..." + os.environ["OPENAI_API_KEY"][-4:])

OPENAI_API_KEY exists: True
Length: 164
Preview: sk-proj...8CoA


In [10]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

hr_policy_docs = {
    "sick_leave_policy.txt": """
    Employees receive 8 paid sick days per calendar year.
    Sick leave may be used for illness, medical appointments, or caring for an immediate family member.
    Employees should notify their manager as soon as possible when taking sick leave.
    """,

    "remote_work_policy.txt": """
    Eligible employees may work remotely up to two days per week.
    Remote work requires manager approval before the remote work date.
    Employees must remain available during normal working hours and maintain productivity expectations.
    """,

    "vacation_policy.txt": """
    Vacation requests must be submitted at least two weeks in advance.
    Approval depends on staffing levels, business needs, and manager review.
    Employees should not finalize travel plans until vacation approval is confirmed.
    """,

    "overtime_policy.txt": """
    Overtime must be approved by a supervisor before extra hours are worked.
    Employees who work overtime without approval may be asked to provide justification.
    Approved overtime must be recorded in the timekeeping system.
    """,

    "data_security_policy.txt": """
    Employees must report suspected phishing emails, suspicious links, or possible data breaches immediately.
    Security incidents should be escalated to IT or the security team.
    Employees should not forward suspicious links or download unknown attachments.
    """
}

doc_names = list(hr_policy_docs.keys())
doc_texts = list(hr_policy_docs.values())

vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(doc_texts)

In [11]:
external_context_docs = {
    "hr_automation_context.txt": """
    HR automation systems help reduce repetitive manual work by classifying employee requests,
    retrieving relevant policy information, routing approval cases, and escalating urgent issues.
    Common HR automation actions include drafting emails, creating manager approval tasks,
    logging requests, and sending notifications.
    """,

    "remote_work_context.txt": """
    Remote work workflows commonly require clear eligibility rules, manager approval,
    employee availability during normal working hours, and documented approval records.
    Automated systems can route remote work requests to managers for review.
    """,

    "cybersecurity_context.txt": """
    Security incident workflows require fast escalation when an employee reports phishing,
    suspicious links, data breaches, or possible account compromise. Automated alerts to IT
    or security teams can reduce response time and support incident tracking.
    """,

    "leave_policy_context.txt": """
    Leave policy workflows are commonly improved by automatic request logging, employee guidance,
    manager approval routing, and clear communication of available leave balances or policy rules.
    """
}

external_doc_names = list(external_context_docs.keys())
external_doc_texts = list(external_context_docs.values())

external_vectorizer = TfidfVectorizer(stop_words="english")
external_doc_matrix = external_vectorizer.fit_transform(external_doc_texts)

def retrieve_external_context(query: str, top_k: int = 2) -> str:
    """
    Retrieve supporting external/business context from a second knowledge source.
    This replaces live DuckDuckGo search to avoid DDGS runtime failures.
    """
    query_vector = external_vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, external_doc_matrix).flatten()

    top_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "source": external_doc_names[idx],
            "score": round(float(similarities[idx]), 3),
            "excerpt": external_context_docs[external_doc_names[idx]].strip()
        })

    return json.dumps(results, indent=2)

print("External context retriever created successfully.")

External context retriever created successfully.


In [17]:
from agno.agent import Agent
from agno.team import Team
from agno.models.openai import OpenAIChat
import json

MODEL_ID = "gpt-4o-mini"

def retrieve_hr_policy(query: str, top_k: int = 2) -> str:
    """
    Retrieve relevant internal HR policy documents based on a query.
    """
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, doc_matrix).flatten()

    top_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "source": doc_names[idx],
            "score": round(float(similarities[idx]), 3),
            "excerpt": hr_policy_docs[doc_names[idx]].strip()
        })
    return json.dumps(results, indent=2)

def decide_zapier_action(policy_evidence: str, external_context: str, employee_request: str) -> str:
    """
    Placeholder function to decide a Zapier-style automation action.
    This function would typically contain logic to parse the policy_evidence,
    external_context, and employee_request to determine the best automation.
    """
    if "sick day" in employee_request.lower() or "sick leave" in employee_request.lower():
        return json.dumps({
            "action": "Draft Sick Leave Email Reply",
            "priority": "Medium",
            "reason": "Employee is asking about sick leave policy."
        }, indent=2)
    elif "remote work" in employee_request.lower():
        return json.dumps({
            "action": "Create Remote Work Approval Task",
            "priority": "High",
            "reason": "Employee is requesting to work remotely."
        }, indent=2)
    elif "phishing" in employee_request.lower() or "security" in employee_request.lower():
        return json.dumps({
            "action": "Escalate to IT Security",
            "priority": "Urgent",
            "reason": "Employee reported a potential security incident."
        }, indent=2)
    else:
        return json.dumps({
            "action": "Recommend HR Review",
            "priority": "Low",
            "reason": "No specific automation action identified for the request."
        }, indent=2)


# Agent 1: HR Policy Retriever Agent
policy_agent = Agent(
    name="HR Policy Retriever Agent",
    role="Retrieves internal HR policy evidence",
    model=OpenAIChat(id=MODEL_ID, max_tokens=500),
    tools=[retrieve_hr_policy],
    instructions=[
        "You retrieve relevant internal HR policy excerpts.",
        "Always use the retrieve_hr_policy tool before answering.",
        "Always include the source file name.",
        "Do not answer from memory.",
        "Summarize the retrieved policy evidence clearly."
    ],
    markdown=True,
)

# Agent 2: External Context Retriever Agent
context_agent = Agent(
    name="External Context Retriever Agent",
    role="Retrieves supporting business and workflow context",
    model=OpenAIChat(id=MODEL_ID, max_tokens=500),
    tools=[retrieve_external_context],
    instructions=[
        "Use the retrieve_external_context tool.",
        "Retrieve supporting business, HR automation, remote work, leave, or cybersecurity context.",
        "Do not override internal HR policy.",
        "Keep the response short and focused.",
        "Always include the source file name."
    ],
    markdown=True,
)

# Agent 3: Workflow Decision Agent
workflow_agent = Agent(
    name="Workflow Decision Agent",
    role="Decides whether Zapier-style automation is needed",
    model=OpenAIChat(id=MODEL_ID, max_tokens=500),
    tools=[decide_zapier_action],
    instructions=[
        "Use the decide_zapier_action tool.",
        "Decide whether the employee request needs a Zapier action.",
        "Return the recommended action, priority, and reason.",
        "Do not say the Zapier action was actually completed."
    ],
    markdown=True,
)

# Coordinator Agent / Team
team = Team(
    name="HR Policy Coordinator",
    members=[policy_agent, context_agent, workflow_agent],
    model=OpenAIChat(id=MODEL_ID, max_tokens=1200),
    instructions=[
        "You are the coordinator for an HR Policy Email Triage Assistant.",
        "Coordinate the HR Policy Retriever Agent, External Context Retriever Agent, and Workflow Decision Agent.",
        "Internal HR policy evidence has priority over supporting external context.",
        "If policy evidence is weak or missing, recommend human HR review.",
        "Do not claim an action was completed; only recommend or simulate the action.",
        """
        Final response format:
        1. Employee Request Classification
        2. Retrieved Policy Evidence
        3. Supporting Context
        4. Final HR Answer
        5. Recommended Zapier Action
        6. Business Value
        7. Risk or Failure Case
        """
    ],
    show_members_responses=True,
)

print("Fixed agents and coordinator created successfully.")

Fixed agents and coordinator created successfully.


In [18]:
test_questions = [
    "How many sick days do employees receive?",
    "Can I work remotely next Friday?",
    "I think I clicked a phishing link. What should I do?"
]

for question in test_questions:
    print("=" * 100)
    print("EMPLOYEE QUESTION:", question)
    print("=" * 100)

    team.print_response(
        f"""
        Employee request: {question}

        Please coordinate the HR Policy Retriever Agent, External Context Retriever Agent,
        and Workflow Decision Agent to produce the final structured output.
        """,
        stream=True
    )

EMPLOYEE QUESTION: How many sick days do employees receive?


Output()

EMPLOYEE QUESTION: Can I work remotely next Friday?


Output()

EMPLOYEE QUESTION: I think I clicked a phishing link. What should I do?


Output()